# GenAI-Traces: Privacy & PII Detection

This notebook demonstrates:
- PII detection using regex patterns
- Redaction strategies (template, partial, hash)
- Integrating privacy protection with tracing

In [ ]:
import sys
sys.path.insert(0, '..')

## 1. PII Detection

In [ ]:
from genai_traces.privacy import PIIDetector, PIIMatch

detector = PIIDetector()

# Test text with various PII
text = """
Please contact John at john.doe@example.com or call him at 555-123-4567.
His SSN is 123-45-6789 and his credit card is 4111111111111111.
The server IP is 192.168.1.100.
API key: sk-1234567890abcdefghij
"""

# Detect all PII
matches = detector.detect(text)

print("Detected PII:")
print("-" * 50)
for match in matches:
    print(f"  Type: {match.type:15} Value: {match.value[:30]:30} Confidence: {match.confidence:.2f}")

In [ ]:
# Quick check if text contains PII
test_texts = [
    "Hello, how are you?",
    "Contact me at user@email.com",
    "My phone is 555-123-4567",
    "The weather is nice today.",
]

print("PII Detection Results:")
print("-" * 50)
for text in test_texts:
    has_pii = detector.contains_pii(text)
    pii_types = detector.detect_types(text)
    print(f"  '{text[:40]:40}' -> PII: {has_pii}, Types: {pii_types or 'None'}")

## 2. PII Redaction

In [ ]:
from genai_traces.privacy import Redactor

redactor = Redactor()

text = "Contact john.doe@example.com or call 555-123-4567. SSN: 123-45-6789"
matches = detector.detect(text)

print("Original text:")
print(f"  {text}")
print()

# Template redaction (default)
redacted_template = redactor.redact(text, matches, strategy="template")
print("Template redaction:")
print(f"  {redacted_template}")
print()

# Partial redaction
redacted_partial = redactor.redact(text, matches, strategy="partial")
print("Partial redaction:")
print(f"  {redacted_partial}")
print()

# Hash redaction
redacted_hash = redactor.redact(text, matches, strategy="hash")
print("Hash redaction:")
print(f"  {redacted_hash}")

## 3. Custom PII Patterns

In [ ]:
# Add custom pattern for employee IDs
detector.add_pattern("employee_id", r"EMP-\d{6}")

text_with_emp_id = "Employee EMP-123456 submitted the report."
matches = detector.detect(text_with_emp_id)

print("Custom pattern detection:")
for match in matches:
    print(f"  Type: {match.type}, Value: {match.value}")

## 4. Privacy-Aware Tracing

In [ ]:
from genai_traces import init_tracer
from genai_traces.exporters import ConsoleExporter
from genai_traces.core.types import SpanType

tracer = init_tracer(
    service_name="privacy-demo",
    exporters=[ConsoleExporter(pretty=True)],
)

def trace_with_privacy(prompt: str, completion: str):
    """Trace an LLM call with PII detection and redaction."""
    with tracer.start_as_current_span("privacy_aware_llm", SpanType.LLM) as span:
        # Check for PII in prompt
        prompt_matches = detector.detect(prompt)
        if prompt_matches:
            span.set_attribute("privacy.pii_detected", True)
            span.set_attribute("privacy.pii_types", list({m.type for m in prompt_matches}))
            # Store redacted version
            span.set_attribute("llm.prompt", redactor.redact(prompt, prompt_matches))
            span.set_attribute("privacy.redacted", True)
        else:
            span.set_attribute("privacy.pii_detected", False)
            span.set_attribute("llm.prompt", prompt)
        
        # Check for PII in completion
        completion_matches = detector.detect(completion)
        if completion_matches:
            span.set_attribute("llm.completion", redactor.redact(completion, completion_matches))
        else:
            span.set_attribute("llm.completion", completion)

# Test with PII
trace_with_privacy(
    prompt="Send the report to john@example.com",
    completion="I've sent the report to the specified email address."
)

print("\nPrivacy-aware trace completed!")

## Summary

This notebook demonstrated:
- ✅ PII detection with regex patterns
- ✅ Multiple redaction strategies
- ✅ Custom PII pattern support
- ✅ Privacy-aware tracing integration